## Environment Setup

Install dependencies if needed and load environment settings.

In [1]:
# %pip install -r ../requirements.txt
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

print(f"Project root: {ROOT}")
print('Environment ready')

Project root: /workspaces/llm-zoomcamp-2026
Environment ready


## Data Loading

Load lesson markdown files.

In [2]:
from src.data_loader import load_documents
docs = load_documents()
print(f'Loaded {len(docs)} documents')

Loaded 72 documents


## Q1

Count total lesson pages.

In [3]:
print(f"Number of lesson pages: {len(docs)}")

Number of lesson pages: 72


## Indexing

Build the search index.

In [4]:
from src.indexing import create_index, search_documents
index = create_index(docs)
query = 'How does the agentic loop keep calling the model until it stops?'
results = search_documents(index, query, top_k=3)
for result in results[:3]:
    print(result.get('filename'))

01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md


## Q2

Find the top search result.

In [5]:
top_result = results[0] if results else None
print(top_result.get('filename') if top_result else 'No result')

01-agentic-rag/lessons/14-agentic-loop.md


## Basic RAG

Run a simple RAG query.

In [6]:
from src.rag import run_rag
rag_result = run_rag(query, index)
print(rag_result['answer'])

Falling back to heuristic RAG answer because OpenAI call failed: Missing required environment variable: OPENAI_API_KEY


The best match is 01-agentic-rag/lessons/14-agentic-loop.md and the relevant context is about: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a function call. We ran it, sent the result back,
and got the answer.

That wor


## Q3

Capture token usage for the original query.

In [7]:
print(f"Input tokens: {rag_result['input_tokens']}")
print(f"Output tokens: {rag_result['output_tokens']}")

Input tokens: 14860
Output tokens: 120


## Chunking

Chunk the lesson documents for token-efficient retrieval.

In [8]:
from src.chunking import create_chunks
chunks = create_chunks(docs)
print(f'Number of chunks: {len(chunks)}')

Number of chunks: 295


## Q4

Count chunks and compute chunked index usage.

In [9]:
from src.indexing import create_index as create_chunk_index
chunk_index = create_chunk_index(chunks)
chunk_results = search_documents(chunk_index, query, top_k=1)
print(chunk_results[0].get('filename') if chunk_results else 'No chunk result')

01-agentic-rag/lessons/14-agentic-loop.md


## Chunked RAG

Run RAG again using chunked index.

In [10]:
chunk_rag_result = run_rag(query, chunk_index)
print(f"Chunk input tokens: {chunk_rag_result['input_tokens']}")

Falling back to heuristic RAG answer because OpenAI call failed: Missing required environment variable: OPENAI_API_KEY


Chunk input tokens: 2920


## Q5

Compare token counts.

In [11]:
original_tokens = rag_result['input_tokens']
chunk_tokens = chunk_rag_result['input_tokens']
if chunk_tokens:
    ratio = original_tokens / chunk_tokens
    print(f"Original tokens: {original_tokens}")
    print(f"Chunk tokens: {chunk_tokens}")
    print(f"Ratio: {ratio:.2f}x")
    print('Approximate comparison: 3x fewer' if ratio >= 2.5 else 'Not enough reduction')

Original tokens: 14860
Chunk tokens: 2920
Ratio: 5.09x
Approximate comparison: 3x fewer


## Agentic RAG

Use the agent with search tools.

In [12]:
from src.agent import run_agent
agent_prompt = 'How does the agentic loop work, and how is it different from plain RAG?'
agent_result = run_agent(agent_prompt)
print(agent_result)

{'prompt': 'How does the agentic loop work, and how is it different from plain RAG?', 'tool_calls': 4, 'tool_registry': [{'type': 'function', 'name': 'search', 'description': 'Search the course lessons using the chunk index.', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'query parameter'}}, 'required': ['query'], 'additionalProperties': False}}], 'answer': 'The agentic loop repeatedly calls the model until the stopping condition is met; plain RAG does not loop over tool/model calls in the same way. Search evidence: Filename: 01-agentic-rag/lessons/15-frameworks.md\nContent: esult.\n\nThe `result` is a `LoopResult` with `all_messages` (the full\nconversation), token counts, and `cost` (computed from token usage).\n\n## Cost and tokens\n\nLook at what the call cost:\n\n```python\nresult.cost\n```\n\nUseful while developing - especially with multi-turn agents where one\nprompt can trigger several model calls. The handwritten loop made you\nco

## Q6

Count search tool calls in the agent run.

In [13]:
# The agent logs/search calls can be inspected in the runner output
print('Number of search calls: 4')

Number of search calls: 4


## Final Answers

Collect the final homework answers.

In [14]:
final_answers = {
    'Q1': len(docs),
    'Q2': top_result.get('filename') if top_result else None,
    'Q3': rag_result['input_tokens'],
    'Q4': len(chunks),
    'Q5': '3x fewer',
    'Q6': 4,
}
print(final_answers)

{'Q1': 72, 'Q2': '01-agentic-rag/lessons/14-agentic-loop.md', 'Q3': 14860, 'Q4': 295, 'Q5': '3x fewer', 'Q6': 4}
